# Tugas Kelompok Week 8
## Optimasi Model CNN dan Implementasi Aplikasi Sederhana

Notebook ini digunakan untuk membuat model CNN pada dataset CIFAR-10, melakukan optimasi sederhana menggunakan hyperparameter tuning, menambahkan Batch Normalization dan Dropout, melakukan evaluasi akhir, serta menyimpan model untuk digunakan pada aplikasi Streamlit.

## 1. Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

## 2. Load Dataset dan Preprocessing

Dataset yang digunakan adalah CIFAR-10. Dataset ini memiliki 10 kelas gambar dengan ukuran 32 x 32 pixel dan 3 channel warna RGB.

In [ ]:
# Load dataset CIFAR-10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Normalisasi pixel dari 0-255 menjadi 0-1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# One-hot encoding label
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Jumlah data train:", X_train.shape)
print("Jumlah data test :", X_test.shape)
print("Jumlah kelas     :", num_classes)

## 3. Menampilkan Contoh Dataset

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i])
    plt.title(class_names[int(y_train[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Fungsi Membuat Model CNN

Model menggunakan Conv2D, BatchNormalization, MaxPooling2D, Dropout, dan Dense. Batch normalization digunakan untuk membantu proses training lebih stabil, sedangkan dropout digunakan untuk mengurangi overfitting.

In [ ]:
def build_cnn_model(filters=(32, 64, 128), dense_units=128, dropout_rate=0.5, learning_rate=0.001):
    model = Sequential([
        Input(shape=(32, 32, 3)),

        Conv2D(filters[0], (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(filters[1], (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(filters[2], (3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(dense_units, activation="relu"),
        Dropout(dropout_rate),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

## 5. Hyperparameter Tuning Sederhana

Pada bagian ini dilakukan percobaan beberapa kombinasi hyperparameter. Parameter yang diuji adalah jumlah filter, jumlah neuron Dense, dropout rate, dan learning rate.

In [ ]:
configs = [
    {"name": "Model A", "filters": (32, 64, 128), "dense_units": 128, "dropout_rate": 0.4, "learning_rate": 0.001},
    {"name": "Model B", "filters": (64, 128, 256), "dense_units": 256, "dropout_rate": 0.5, "learning_rate": 0.001},
    {"name": "Model C", "filters": (32, 64, 128), "dense_units": 256, "dropout_rate": 0.3, "learning_rate": 0.0005},
]

TUNING_EPOCHS = 3
BATCH_SIZE = 64

results = []

for cfg in configs:
    print("Training", cfg["name"])
    model_temp = build_cnn_model(
        filters=cfg["filters"],
        dense_units=cfg["dense_units"],
        dropout_rate=cfg["dropout_rate"],
        learning_rate=cfg["learning_rate"]
    )

    history_temp = model_temp.fit(
        X_train,
        y_train_cat,
        epochs=TUNING_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test_cat),
        verbose=1
    )

    val_acc = max(history_temp.history["val_accuracy"])
    results.append({"config": cfg, "val_accuracy": val_acc})
    print(cfg["name"], "Validation Accuracy Terbaik:", round(val_acc, 4))
    print("-" * 50)

## 6. Menentukan Model Terbaik

In [ ]:
best_result = max(results, key=lambda x: x["val_accuracy"])
best_config = best_result["config"]

print("Konfigurasi terbaik:")
print(best_config)
print("Validation accuracy terbaik:", round(best_result["val_accuracy"], 4))

## 7. Training Model Final

Model final dibuat menggunakan konfigurasi terbaik dari hasil tuning.

In [ ]:
FINAL_EPOCHS = 10

final_model = build_cnn_model(
    filters=best_config["filters"],
    dense_units=best_config["dense_units"],
    dropout_rate=best_config["dropout_rate"],
    learning_rate=best_config["learning_rate"]
)

final_model.summary()

history = final_model.fit(
    X_train,
    y_train_cat,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test_cat),
    verbose=1
)

## 8. Evaluasi Akhir Model

In [ ]:
loss, accuracy = final_model.evaluate(X_test, y_test_cat, verbose=0)

print("Hasil Evaluasi Akhir")
print(f"Loss    : {loss:.4f}")
print(f"Akurasi : {accuracy:.2%}")

## 9. Grafik Akurasi dan Loss

In [ ]:
epochs = range(1, len(history.history["accuracy"]) + 1)

plt.plot(epochs, history.history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(epochs, history.history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("Grafik Akurasi Model CNN")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(epochs)
plt.legend()
plt.show()

plt.plot(epochs, history.history["loss"], marker="o", label="Training Loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation Loss")
plt.title("Grafik Loss Model CNN")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()

## 10. Testing Prediksi pada Data Test

In [ ]:
predictions = final_model.predict(X_test[:10], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = y_test[:10].flatten()

plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i])
    plt.title(f"Asli: {class_names[true_labels[i]]}
Pred: {class_names[predicted_labels[i]]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 11. Simpan Model

Model disimpan agar dapat digunakan pada aplikasi Streamlit.

In [ ]:
final_model.save("cnn_cifar10_optimized.keras")
print("Model berhasil disimpan dengan nama cnn_cifar10_optimized.keras")

## 12. Catatan Implementasi Aplikasi

Setelah file model berhasil dibuat, gunakan file `app.py` untuk menjalankan aplikasi Streamlit. File model `cnn_cifar10_optimized.keras` harus berada dalam folder yang sama dengan `app.py`.